# 第16周 · 练习：欧拉法与 Flow Matching 的四个零件

Flow Matching 的全部代码可以拆成四个零件：**欧拉积分器**（采样）、**线性插值**、**条件速度目标**、**损失 + 一个训练步**。今天把它们逐个写出来，每个零件都有一个"事先知道正确答案"的自查——这正是第 9 周诊断配方的精神：先算出期望值，再看实际值。

约定与讲义相同（MIT 6.S184 讲义 §2.1 Algorithm 1、§3.3 Algorithm 3）：$x_0$ 是噪声、$x_1$ 是数据，$x_t = (1 - t)\,x_0 + t\,x_1$，**$t = 0$ 是噪声、$t = 1$ 是数据**。

In [ ]:
# 运行这一格来安装课程工具包（每次在 Colab 打开 notebook 后运行一次）
import importlib.util, subprocess, sys
if importlib.util.find_spec("zero2researcher") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ReaganGen/Zero2Researcher"], check=True)
print("课程工具包已就绪")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from zero2researcher import data, experiments, generative
from zero2researcher.checks import week16 as ck

device = torch.device("cpu")      # 全部是 2D 小实验，固定用 CPU
print("torch", torch.__version__, "| device:", device)

## E1 · 欧拉积分器

写 `euler_integrate(u, x0, n_steps, t_start=0.0, t_end=1.0)`：步长 $h = (t_\text{end} - t_\text{start}) / n$，第 $i$ 步（$i = 0, 1, \dots, n-1$）的时刻是 $t_\text{start} + i\,h$，更新 $x \leftarrow x + h \cdot u(x, t)$。返回**整条轨迹**：把起点和每一步之后的 $x$ 都存下来，`torch.stack` 成形状 `(n_steps + 1, *x0.shape)` 的张量。

自查会用三个有解析解的速度场检查你：常速度场、$u = -x$（解为 $x_0 e^{-t}$）、以及依赖时间的 $u = (t, 0)$。

In [ ]:
def euler_integrate(u, x0, n_steps, t_start=0.0, t_end=1.0):
    # 在这里写你的代码
    raise NotImplementedError


ck.euler_integrate(euler_integrate)

**E1b · 欧拉法的误差随步数怎样变？** 对 $u(x, t) = -x$、起点 $x_0 = 1$（形状 `(1, 1)` 的张量），从 $t = 0$ 积分到 $t = 1$，精确解是 $e^{-1}$。对 $n \in \{1, 2, 4, \dots, 1024\}$ 算出终点与 $e^{-1}$ 的绝对误差，存进列表 `errors`。下面的代码会画双对数图并拟合斜率。

In [ ]:
ns = [2 ** k for k in range(11)]
x_start = torch.ones(1, 1, dtype=torch.float64)
# 在这里写你的代码
raise NotImplementedError
slope = np.polyfit(np.log(ns), np.log(errors), 1)[0]
for n, e in zip(ns, errors):
    print(f"n = {n:5d}  误差 {e:.2e}")
print(f"双对数图上的斜率 ≈ {slope:.2f}")

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.loglog(ns, errors, "o-", label="Euler error")
ax.loglog(ns, errors[0] * ns[0] / np.array(ns), "--", color="gray", label="Reference: error ∝ 1/n")
ax.set_xlabel("Number of Euler steps n")
ax.set_ylabel("|x(1) − exp(−1)|")
ax.set_title("Euler is first-order: doubling the steps halves the error")
ax.legend()
plt.show()

**问题**（一两句话）：步数翻倍，误差变成原来的几分之一？斜率说明了什么？如果生成一张图要求误差小 10 倍，采样时间要变成几倍？

（在这里回答）

## E2 · 线性插值与条件速度目标

写 `interpolant(x0, x1, t)`（$t$ 的形状是 `(B, 1)`，要自动广播到 `(B, 2)`）和 `fm_target(x0, x1)`。

In [ ]:
def interpolant(x0, x1, t):
    # 在这里写你的代码
    raise NotImplementedError


def fm_target(x0, x1):
    # 在这里写你的代码
    raise NotImplementedError


ck.interpolant(interpolant)
ck.fm_target(fm_target)

**E2b · "高斯插值"是什么意思？** 固定一个数据点 $x_1 = (2, -1)$，配上 10 万个不同的噪声 $x_0$，取 $t = 0.25$。先**在纸上**写出 $x_t$ 的均值与标准差（每个坐标），再用代码验证：把 `xt` 的样本均值与样本标准差算出来。

In [ ]:
torch.manual_seed(0)
x1_fixed = torch.tensor([2.0, -1.0]).repeat(100_000, 1)
x0_many = torch.randn_like(x1_fixed)
t_q = torch.full((100_000, 1), 0.25)
# 在这里写你的代码
raise NotImplementedError
print("x_t 的样本均值:", mean_xt.numpy().round(3), "| 样本标准差:", std_xt.numpy().round(3))

（在这里写下你在纸上算的均值与标准差，并说明为什么叫"高斯插值"）

## E3 · Flow Matching 损失

写 `fm_loss(model, x1)`：在函数**里面**抽噪声 `x0 = torch.randn_like(x1)` 与时间 `t = torch.rand(B, 1)`，算插值 $x_t$，返回 `((model(x_t, t) - 目标) ** 2).mean()`。

自查里藏着一个解析解：如果数据**只有一个点** $c$，最优速度场是讲义 S1 里的 $u^*(x, t) = (c - x)/(1 - t)$——把它当成 `model` 传进去，你的损失应该**恰好接近 0**。插值、目标、时间方向只要有一处不配套，这个检查就过不了。

In [ ]:
def fm_loss(model, x1):
    # 在这里写你的代码
    raise NotImplementedError


ck.fm_loss(fm_loss)

**问题**（两三句话）：数据只有一个点时最优损失是 0；数据有两个点（例如 $(-3, 0)$ 与 $(3, 0)$ 各一半）时，最优损失还能是 0 吗？为什么？这对"用训练损失判断样本质量"意味着什么？

（在这里回答）

## E4 · 一个训练步

下面的 `VelocityMLP` 与讲义相同（已给出）。写 `train_step(model, optimizer, x1)`：第 8 周的五步核心——清零梯度、算 `fm_loss`、反向传播、更新参数——并返回这一步损失的 Python 数（`.item()`）。自查会用一个全新的小网络跑 300 步，看损失是否明显下降。

In [ ]:
class VelocityMLP(nn.Module):
    def __init__(self, hidden=128, depth=2):
        super().__init__()
        layers, d = [], 3
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.SiLU()]
            d = hidden
        layers.append(nn.Linear(d, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x, t):
        t = torch.as_tensor(t, dtype=x.dtype).reshape(-1, 1).expand(len(x), 1)
        return self.net(torch.cat([x, t], dim=1))


def train_step(model, optimizer, x1):
    # 在这里写你的代码
    raise NotImplementedError


ck.train_step(train_step)

把四个零件拼起来：在 8 个小高斯组成的圆环上训练 2000 步，再用你的 `euler_integrate` 采样（代码已给出，跑通即可）。

In [ ]:
def ring(n):
    return generative.sample_gaussian_mixture(n, k=8, radius=3.0, std=0.3)


experiments.set_seed(0)
model = VelocityMLP()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = [train_step(model, optimizer, ring(512)) for _ in range(2000)]
with torch.no_grad():
    samples = euler_integrate(model, torch.randn(2000, 2, generator=torch.Generator().manual_seed(1)), 100)[-1]
torch.manual_seed(2)
ref = ring(2000)
print(f"能量距离 {generative.energy_distance(samples, ref):.4f} | k-NN precision {generative.knn_precision(samples, ref):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(np.convolve(losses, np.ones(50) / 50, mode="valid"))
axes[0].set_xlabel("Step (moving average of 50)")
axes[0].set_ylabel("FM loss")
axes[0].set_title("Loss drops, then plateaus above zero")
axes[1].scatter(ref[:, 0], ref[:, 1], s=2, color="lightgray", label="Real")
axes[1].scatter(samples[:, 0], samples[:, 1], s=2, label="Generated")
axes[1].set_aspect("equal")
axes[1].legend(markerscale=4)
axes[1].set_title("Four small functions are a whole generative model")
plt.tight_layout()
plt.show()

**复盘**（两三句话）：四个零件里，哪一个写错时**不会报错**、只会让结果悄悄变差？如果你只能加一行检查代码来提前发现它，你会加什么？

（在这里回答）